In [9]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

import matplotlib.pyplot as plt
from IPython.display import display

import numpy as np
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

In [10]:
loader = HFDatasetLoader(
    dataset_name="shijiabraham/insurance-data",
    split="train",
    filename="datos/insurance_data_ml_regressionlineal.csv",
)

In [11]:
df = loader.load()
df.head(5)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 1. Evidencia de preprocesamiento

Se analiza la calidad de los datos del dataset Medical Insurance
para identificar valores faltantes, registros duplicados y tipos de
datos antes de entrenar los modelos.

In [12]:
# Copia de los datos originales
df_original = df.copy(deep=True)

print("Dimensiones iniciales:", df_original.shape)

print("\nTipos de datos:")
print(df_original.dtypes)

print("\nValores faltantes:")
print(df_original.isnull().sum())

print("\nRegistros duplicados:")
print(df_original.duplicated().sum())

print("\nPrimeros registros:")
display(df_original.head())

Dimensiones iniciales: (1338, 7)

Tipos de datos:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

Valores faltantes:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Registros duplicados:
1

Primeros registros:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [13]:
# Crear copia para realizar la limpieza
df_limpio = df_original.copy(deep=True)

# Estado inicial
filas_antes = len(df_limpio)
duplicados_antes = df_limpio.duplicated().sum()
nulos_antes = df_limpio.isnull().sum().sum()

# Eliminar duplicados exactos
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)

# Estado final
filas_despues = len(df_limpio)
duplicados_despues = df_limpio.duplicated().sum()
nulos_despues = df_limpio.isnull().sum().sum()

# Comparación de resultados
comparacion = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "Registros duplicados",
        "Valores faltantes"
    ],
    "Antes": [
        filas_antes,
        duplicados_antes,
        nulos_antes
    ],
    "Después": [
        filas_despues,
        duplicados_despues,
        nulos_despues
    ]
})

display(comparacion)

,Indicador,Antes,Después
0,Total de registros,1338,1337
1,Registros duplicados,1,0
2,Valores faltantes,0,0


### Justificación de las operaciones de preprocesamiento

Se realizó una revisión inicial para identificar valores faltantes, registros duplicados y tipos de datos del dataset Medical Insurance.

Se decidió eliminar los registros duplicados exactos para evitar la repetición de observaciones durante el entrenamiento y la evaluación de los modelos.

No fue necesario imputar valores faltantes, ya que en la revisión inicial no se identificaron datos nulos.

Las variables numéricas utilizadas para la regresión fueron `bmi`, `age` y `charges`. Además, la clase `RegresionLineal` implementa el método `preparar_datos()`, que permite seleccionar las variables, convertirlas a valores numéricos y descartar registros inválidos.

Para la clasificación se realizará posteriormente la codificación de las variables categóricas, utilizando únicamente los datos de entrenamiento para ajustar las transformaciones necesarias.

## 2. Modelo no Supervisado K- Means 



### Preprocesamiento y Transformación de Datos

Separación de columnas: Se dividen las variables en cuantitativas (num_cols) y cualitativas (cat_cols).

StandardScaler(): Escala las variables continuas para que tengan media 0 y desviación estándar 1. Esto es crítico en K-Means, ya que si charges varía entre $1.000 y $60.000 y age entre 18 y 64, la variable charges dominaría por completo el cálculo de distancias euclidianas.

OneHotEncoder(drop='first'): Transforma las variables categóricas (texto) en binarias (0 o 1). El argumento drop='first' elimina la primera categoría de cada variable para evitar la trampa de la multicolinealidad (redundancia numérica).

ColumnTransformer: Aplica ambas transformaciones simultáneamente y devuelve la matriz X_prep, lista para ser procesada por el algoritmo.

In [14]:
# ---------------------------------------------------------
# 2. PREPROCESAMIENTO Y TRANSFORMACIÓN DE DATOS
# ---------------------------------------------------------
num_cols = ['age', 'bmi', 'children', 'charges']
cat_cols = ['sex', 'smoker', 'region']

# ColumnTransformer: Estandariza continuas y aplica One-Hot Encoding a categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ]
)

X_prep = preprocessor.fit_transform(df)

### Determinación del Número Óptimo de Clústeres (K)

Bucle de evaluación: Ejecuta el algoritmo K-Means variando K entre 2 y 10.

n_init=10 y random_state=42: Garantiza la reproducibilidad y ejecuta el algoritmo 10 veces con diferentes semillas iniciales en cada valor de K, conservando el resultado con menor inercia.

inertia_ (Inercia intraclúster / WCSS): Suma de distancias al cuadrado de los puntos a su centroide. Representa qué tan compactos son los grupos.silhouette_score(): Mide el grado de separación entre clústeres. Toma valores entre -1 (mala asignación) y 1 (excelente separación).

In [15]:
# ---------------------------------------------------------
# 3. DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLÚSTERES (K)
# ---------------------------------------------------------
wcss = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_eval.fit(X_prep)
    wcss.append(kmeans_eval.inertia_)
    silhouette_scores.append(silhouette_score(X_prep, kmeans_eval.labels_))

### Graficas de codo y silueta

Gráficas de Codo y Silueta: Visualizan ambas métricas en paralelo para identificar dónde disminuye drásticamente la inercia (codo) manteniendo una silueta alta.

In [16]:
# Gráfico 1: Método del Codo y Puntaje de Silueta
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(k_range, wcss, 'bo-', linewidth=2, markersize=8)
ax[0].set_title('Método del Codo (Elbow Method)', fontsize=13, fontweight='bold')
ax[0].set_xlabel('Número de Clústeres (K)')
ax[0].set_ylabel('WCSS (Inercia Intraclúster)')
ax[0].grid(True, linestyle='--', alpha=0.6)

ax[1].plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax[1].set_title('Puntaje de Silueta (Silhouette Score)', fontsize=13, fontweight='bold')
ax[1].set_xlabel('Número de Clústeres (K)')
ax[1].set_ylabel('Silhouette Score')
ax[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('datos/graficas/elbow_silhouette.png', dpi=300)
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x73febc1c7110> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 1400x500 with 2 Axes>

###  Entrenamiento del Modelo Final (K = 3)

fit_predict(X_prep): Tras definir K=3 a partir de la evidencia del paso anterior, ajusta el modelo a la matriz estandarizada X_prep y asigna a cada fila del dataframe original un identificador de clúster (0, 1 o 2).

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_prep)

### Cálculo de Métricas de Evaluación No Supervisada

kmeans.inertia_: Extrae la compacidad total del modelo ajustado (K=3).

silhouette_score(): Calcula la métrica de silueta global promedio para verificar la cohesión y separación de las 3 agrupaciones.

davies_bouldin_score(): Evalúa la similitud promedio entre cada clúster y su más parecido. En esta métrica, un valor más bajo indica una mejor separación entre grupos.

In [ ]:
inertia = kmeans.inertia_
sil_score = silhouette_score(X_prep, df['Cluster'])
db_index = davies_bouldin_score(X_prep, df['Cluster'])

### Reducción de Dimensionalidad y Visualización 2D con PCA

PCA(n_components=2): Dado que los datos procesados tienen 9 dimensiones (tras el One-Hot Encoding), la mente humana no puede visualizarlos directamente. PCA (Análisis de Componentes Principales) comprime esas 9 dimensiones en 2 ejes sintéticos (PCA1 y PCA2) conservando la máxima varianza posible del espacio original.

sns.scatterplot: Grafica cada paciente en el plano 2D diferenciando su asignación de clúster por color, lo que permite verificar visualmente el traslape y las fronteras de división.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_prep)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(9, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', palette='Set2', data=df, alpha=0.8, s=70)
plt.title('Visualización 2D de Clústeres mediante PCA', fontsize=13, fontweight='bold')
plt.xlabel(f'PCA Componente 1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
plt.ylabel(f'PCA Componente 2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Clúster')
plt.tight_layout()
plt.savefig('pca_clusters.png', dpi=300)
plt.show()

### Gráficos de Distribución por Característica

plt.subplots(2, 2): Crea una matriz gráfica de 2 filas por 2 columnas.

sns.boxplot: Genera diagramas de caja para comparar la mediana, el rango intercuartílico y los valores atípicos (outliers) de las variables numéricas primarias (charges, age, bmi) entre los 3 clústeres.

sns.barplot: Muestra cómo interactúa el hábito de fumar con los cargos médicos promedio dentro de cada clúster, facilitando la interpretación técnica del comportamiento del modelo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='Cluster', y='charges', data=df, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Cargos Médicos ($) por Clúster', fontweight='bold')

sns.boxplot(x='Cluster', y='age', data=df, ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Edad (Años) por Clúster', fontweight='bold')

sns.boxplot(x='Cluster', y='bmi', data=df, ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Índice de Masa Corporal (BMI) por Clúster', fontweight='bold')

sns.barplot(x='Cluster', y='charges', hue='smoker', data=df, ax=axes[1, 1], palette='Set1', errorbar=None)
axes[1, 1].set_title('Cargos Promedio por Clúster y Hábito de Fumar', fontweight='bold')

plt.tight_layout()
plt.savefig('datos/graficas/cluster_feature_distributions.png', dpi=300)
plt.show()

### Perfilamiento Estadístico y Conteo

groupby('Cluster').mean(): Calcula los promedios reales (en unidades originales como dólares o años) de cada variable cuantitativa por grupo.

pd.crosstab(..., normalize='index') * 100: Genera la tabla de contingencia porcentual para ver la proporción de fumadores y no fumadores en cada clúster.

value_counts(): Cuenta la cantidad exacta de asegurados que quedaron asignados a cada uno de los 3 clústeres.

In [ ]:
print(df.groupby('Cluster')[['age', 'bmi', 'children', 'charges']].mean())
print(pd.crosstab(df['Cluster'], df['smoker'], normalize='index') * 100)
print(df['Cluster'].value_counts())